# Dataset profile

IMP-02. Loads the Kottarathil PCOS dataset and checks the record count, number of features and class balance.

In [1]:
from pathlib import Path

import pandas as pd

# Project root: works whether the notebook is started from notebooks/ or from the project folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "raw" / "PCOS_data_without_infertility.xlsx"

pd.set_option("display.max_rows", 60)
print("Reading:", DATA_FILE.relative_to(ROOT))

Reading: data\raw\PCOS_data_without_infertility.xlsx


## Sheets in the file

In [2]:
xls = pd.ExcelFile(DATA_FILE)
xls.sheet_names

['Instructions', 'Full_new']

## Load the data

In [3]:
df = pd.read_excel(DATA_FILE, sheet_name="Full_new")
n_records, n_columns = df.shape
print(f"Records: {n_records}")
print(f"Columns: {n_columns}")

Records: 541
Columns: 45


## Column inventory

Column names are printed with quotes so leading/trailing spaces show up. Each column is tagged as identifier, label, stray or feature.

In [4]:
ID_COLS = ["Sl. No", "Patient File No."]
LABEL = "PCOS (Y/N)"
STRAY = [c for c in df.columns if str(c).startswith("Unnamed")]

def role(col):
    if col in ID_COLS:
        return "identifier"
    if col == LABEL:
        return "label"
    if col in STRAY:
        return "stray"
    return "feature"

inventory = pd.DataFrame({
    "column": [repr(c) for c in df.columns],
    "role": [role(c) for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "non_missing": df.notna().sum().values,
    "unique_values": df.nunique().values,
    "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
inventory

,column,role,dtype,non_missing,unique_values,example
0,'Sl. No',identifier,int64,541,541,1
1,'Patient File No.',identifier,int64,541,541,1
2,'PCOS (Y/N)',label,int64,541,2,0
3,' Age (yrs)',feature,int64,541,29,28
4,'Weight (Kg)',feature,float64,541,117,44.6
5,'Height(Cm) ',feature,float64,541,50,152.0
6,'BMI',feature,float64,541,355,19.3
7,'Blood Group',feature,int64,541,8,15
8,'Pulse rate(bpm) ',feature,int64,541,11,78
9,'RR (breaths/min)',feature,int64,541,8,22


## Feature count

In [5]:
role_counts = inventory["role"].value_counts().reindex(["identifier", "label", "feature", "stray"], fill_value=0)
role_counts.to_frame("columns")

,columns
role,
identifier,2
label,1
feature,41
stray,1


## Class balance

In [6]:
counts = df[LABEL].value_counts().sort_index()
balance = pd.DataFrame({
    "class": ["Not PCOS (0)", "PCOS (1)"],
    "records": counts.values,
    "percent": (100 * counts / counts.sum()).round(1).values,
})
majority = balance["percent"].max()
print(f"A model that always predicts the majority class would score {majority}% accuracy.")
balance

A model that always predicts the majority class would score 67.3% accuracy.


,class,records,percent
0,Not PCOS (0),364,67.3
1,PCOS (1),177,32.7


## Observations

- The file has 541 patient records and 45 columns. 2 are IDs, 1 is the PCOS label and 1 is an extra empty column (Unnamed: 44), so there are 41 features.
- 177 patients have PCOS (32.7%) and 364 don't (67.3%), so the classes aren't balanced. A model that always says "no PCOS" would still get 67.3% accuracy, so accuracy alone isn't enough. I'll use stratified cross-validation and also look at recall, specificity, F1 and AUC.
- Some column names have extra spaces, like ' Age (yrs)' and 'Height(Cm) ', and "abortions" is spelt "aborptions". I need to clean the names in code so they don't cause errors later.
- AMH and beta-HCG II are stored as text even though they should be numbers, so there must be some non-number values in them. I check this in the next notebook.
- Marriage years and fast food each have one missing value.